# Build Elliot Performance Summary

Đọc tất cả file `{Model}_cutoff_{N}.tsv` trong `results/elliot/*/performance/`,
tổng hợp thành Excel 4 sheets:

| Sheet | Nội dung |
|---|---|
| `long_all_results` | Tất cả metrics, long format, 1 row = 1 (experiment × model × cutoff) |
| `wide_summary` | Mỗi row = 1 (experiment × model), metrics@cutoff là columns — dễ so sánh |
| `main_metrics` | Long format, chỉ 5 metric chính (Precision/Recall/nDCG/MAP/MRR) |
| `u1_metrics` | Long format, chỉ 5 metric cold-start (_u1 suffix) |

Chạy toàn bộ notebook để rebuild file Excel từ kết quả hiện có.

In [ ]:
# ── Cấu hình ──────────────────────────────────────────────────────────────────

RESULTS_ROOT   = r"D:/recsys-pipeline/results/elliot"
PROCESSED_ROOT = r"D:/recsys-pipeline/data/processed"   # để đọc k_user / k_item
REPORTS_ROOT   = r"D:/recsys-pipeline/data/reports"     # sparsity summary CSVs
OUTPUT_PATH    = r"D:/recsys-pipeline/results/elliot/_summary/elliot_performance_summary.xlsx"

# Số chữ số thập phân trong Excel
ROUND_DIGITS = 6

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd

MAIN_METRICS = ["Precision", "Recall", "nDCG", "MAP", "MRR"]
U1_METRICS   = ["Precision_u1", "Recall_u1", "nDCG_u1", "MAP_u1", "MRR_u1"]
ALL_METRICS  = MAIN_METRICS + U1_METRICS

SPARSITY_COLS = [
    "n_interactions_x1000", "oss_pct", "uss", "iss",
    "user_gini", "item_gini",
    "coldstart_user_pct", "coldstart_item_pct",
]

# Thứ tự ưu tiên khi sort strategy
_STRATEGY_ORDER = {"base": 0, "head": 1, "random": 2, "tail": 3}


def parse_experiment_name(name: str) -> dict:
    """Parse folder name into dataset / strategy / keep_frac.

    'hm_random_keep0.05'      -> dataset=hm,     strategy=random, keep_frac=0.05
    'amazon_head_keep0.5'     -> dataset=amazon, strategy=head,   keep_frac=0.5
    'hm_dedup_base_split'     -> dataset=hm,     strategy=base,   keep_frac=1.0
    'amazon_dedup_base_split' -> dataset=amazon, strategy=base,   keep_frac=1.0
    """
    m = re.match(r'^(\w+?)_(random|head|tail)_keep([\d.]+)$', name)
    if m:
        return {"dataset": m.group(1), "strategy": m.group(2), "keep_frac": float(m.group(3))}

    m = re.match(r'^(\w+?)_dedup_base_split$', name)
    if m:
        return {"dataset": m.group(1), "strategy": "base", "keep_frac": 1.0}

    # fallback
    parts = name.split("_")
    return {"dataset": parts[0], "strategy": None, "keep_frac": None}


def load_kcore(exp_name: str, processed_root: Path) -> tuple:
    """Try to read k_user, k_item from data/processed/{exp_name}/metadata.json."""
    mj = processed_root / exp_name / "metadata.json"
    if mj.exists():
        try:
            data = json.loads(mj.read_text(encoding="utf-8"))
            return data.get("k_user"), data.get("k_item")
        except Exception:
            pass
    return None, None


def load_dataset_reports(reports_root) -> pd.DataFrame:
    """Load all {dataset}_{strategy}_summary.csv files from data/reports/.

    Filename format: amazon_random_summary.csv -> dataset=amazon, strategy=random

    Also synthesizes base rows (strategy='base', keep_frac=1.0) from the
    keep_frac=1.0 entry of any existing strategy file for each dataset, so that
    {dataset}_dedup_base_split experiments can be matched.

    Returns DataFrame with columns:
        dataset, strategy, keep_frac,
        n_interactions_x1000, oss_pct, uss, iss,
        user_gini, item_gini, coldstart_user_pct, coldstart_item_pct
    """
    root = Path(reports_root)
    dfs = []
    for csv_path in sorted(root.glob("*_summary.csv")):
        m = re.match(r'^(\w+)_(random|head|tail|base)_summary\.csv$', csv_path.name)
        if not m:
            print(f"[WARN] Skipping unrecognized report file: {csv_path.name}")
            continue
        dataset  = m.group(1)
        strategy = m.group(2)

        try:
            df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"[WARN] Cannot read {csv_path.name}: {e}")
            continue

        df["dataset"]   = dataset
        df["strategy"]  = strategy
        df["keep_frac"] = df["keep_frac"].astype(float)

        keep_cols = ["dataset", "strategy", "keep_frac"] + [
            c for c in SPARSITY_COLS if c in df.columns
        ]
        dfs.append(df[keep_cols])

    if not dfs:
        print("[WARN] No report files found in", reports_root)
        return pd.DataFrame()

    combined = pd.concat(dfs, ignore_index=True)

    # Synthesize base rows: for each dataset, copy keep_frac=1.0 row and
    # relabel strategy='base', so base-split experiments can be matched.
    base_rows = []
    for dataset, grp in combined.groupby("dataset"):
        row_10 = grp[grp["keep_frac"] == 1.0].head(1).copy()
        if row_10.empty:
            print(f"[WARN] No keep_frac=1.0 row for dataset={dataset} — cannot create base row")
            continue
        row_10["strategy"] = "base"
        base_rows.append(row_10)

    if base_rows:
        combined = pd.concat(
            [combined, pd.concat(base_rows, ignore_index=True)],
            ignore_index=True,
        )

    # Drop duplicates (base rows added last, so they win on keep="last")
    combined = combined.drop_duplicates(
        subset=["dataset", "strategy", "keep_frac"], keep="last"
    )
    return combined.reset_index(drop=True)


print("Helpers defined.")

In [ ]:
# ── Collect all results ───────────────────────────────────────────────────────

results_root   = Path(RESULTS_ROOT)
processed_root = Path(PROCESSED_ROOT)

rows = []

for ds_dir in sorted(results_root.iterdir()):
    if not ds_dir.is_dir() or ds_dir.name.startswith("_"):
        continue
    perf_dir = ds_dir / "performance"
    if not perf_dir.exists():
        continue

    exp_meta = parse_experiment_name(ds_dir.name)
    k_user, k_item = load_kcore(ds_dir.name, processed_root)

    for tsv in sorted(perf_dir.glob("*_cutoff_*.tsv")):
        # Match {ModelName}_cutoff_{N}.tsv  (new normalized format)
        m = re.match(r"^(.+)_cutoff_(\d+)\.tsv$", tsv.name)
        if not m:
            continue
        model_short = m.group(1)   # e.g. "ItemKNN", "VSM"
        cutoff      = int(m.group(2))

        try:
            df = pd.read_csv(tsv, sep="\t")
        except Exception as e:
            print(f"[WARN] Cannot read {tsv}: {e}")
            continue

        if df.empty or "model" not in df.columns:
            continue

        for _, row in df.iterrows():
            record = {
                "experiment" : ds_dir.name,
                "dataset"    : exp_meta["dataset"],
                "strategy"   : exp_meta["strategy"],
                "keep_frac"  : exp_meta["keep_frac"],
                "k_user"     : k_user,
                "k_item"     : k_item,
                "cutoff"     : cutoff,
                "model"      : model_short,
                "full_model" : str(row.get("model", "")),
            }
            for metric in ALL_METRICS:
                record[metric] = round(float(row[metric]), ROUND_DIGITS) if metric in row and pd.notna(row[metric]) else float("nan")
            rows.append(record)

long_df = pd.DataFrame(rows)

if long_df.empty:
    raise RuntimeError(f"No performance files found under {RESULTS_ROOT}. "
                       "Run Elliot first: python scripts/run_elliot.py")

# Sort
long_df["_strategy_order"] = long_df["strategy"].map(_STRATEGY_ORDER).fillna(99)
long_df = (
    long_df
    .sort_values(["dataset", "_strategy_order", "keep_frac", "model", "cutoff"])
    .drop(columns=["_strategy_order"])
    .reset_index(drop=True)
)

print(f"Collected {len(long_df)} rows from {long_df['experiment'].nunique()} experiments.")
print(f"Datasets : {sorted(long_df['dataset'].unique())}")
print(f"Models   : {sorted(long_df['model'].unique())}")
print(f"Cutoffs  : {sorted(long_df['cutoff'].unique())}")
long_df.head(6)

In [ ]:
# ── Load & merge sparsity reports ─────────────────────────────────────────────

report_df = load_dataset_reports(REPORTS_ROOT)

if not report_df.empty:
    print(f"Loaded report_df: {len(report_df)} rows")
    print(f"  Datasets  : {sorted(report_df['dataset'].unique())}")
    print(f"  Strategies: {sorted(report_df['strategy'].unique())}")
    print(f"  keep_fracs: {sorted(report_df['keep_frac'].unique())}")
else:
    print("[WARN] report_df is empty — sparsity columns will be missing from output")

# Float-safe merge key (avoids 0.09999... != 0.1 issues)
long_df["_kf_key"] = long_df["keep_frac"].round(6)
report_key = report_df.copy()
report_key["_kf_key"] = report_key["keep_frac"].round(6)

long_df = long_df.merge(
    report_key.drop(columns=["keep_frac"]),
    on=["dataset", "strategy", "_kf_key"],
    how="left",
).drop(columns=["_kf_key"])

# ── Warn on unmatched rows ─────────────────────────────────────────────────────
first_sparsity = SPARSITY_COLS[0] if SPARSITY_COLS else None
if first_sparsity and first_sparsity in long_df.columns:
    unmatched = long_df[long_df[first_sparsity].isna()]
    if not unmatched.empty:
        missing = (
            unmatched[["experiment", "dataset", "strategy", "keep_frac"]]
            .drop_duplicates()
        )
        print(
            f"\n[WARN] {len(unmatched)} performance rows have no sparsity report "
            f"({len(missing)} unique experiments):"
        )
        print(missing.to_string(index=False))
    else:
        print(f"\nAll {len(long_df)} performance rows matched a sparsity report.")

print(f"long_df after merge: {long_df.shape}")

In [ ]:
# ── Sheet: long_all_results ───────────────────────────────────────────────────
# Dữ liệu đầy đủ, long format

col_order_long = [
    "experiment", "dataset", "strategy", "keep_frac", "k_user", "k_item",
    "n_interactions_x1000", "oss_pct", "uss", "iss",
    "user_gini", "item_gini",
    "coldstart_user_pct", "coldstart_item_pct",
    "cutoff", "model", "full_model",
] + ALL_METRICS

sheet_long = long_df[[c for c in col_order_long if c in long_df.columns]].copy()
print(f"long_all_results: {sheet_long.shape}")
sheet_long.head(3)

In [ ]:
# ── Sheet: wide_summary ───────────────────────────────────────────────────────
# Một row = (experiment, model), columns = metric@cutoff
# Chỉ dùng 5 main metrics để bảng không quá rộng.

index_cols = ["experiment", "dataset", "strategy", "keep_frac", "k_user", "k_item", "model"]

wide_df = long_df.pivot_table(
    index=index_cols,
    columns="cutoff",
    values=MAIN_METRICS,
    aggfunc="first",
)
# Flatten MultiIndex columns: ("Precision", 10) -> "Precision@10"
wide_df.columns = [f"{metric}@{cutoff}" for metric, cutoff in wide_df.columns]
wide_df = wide_df.reset_index()

# Attach sparsity cols — constant per experiment, pick first occurrence
sparsity_present = [c for c in SPARSITY_COLS if c in long_df.columns]
if sparsity_present:
    sparsity_by_exp = (
        long_df[["experiment"] + sparsity_present]
        .drop_duplicates(subset=["experiment"])
        .set_index("experiment")
    )
    wide_df = wide_df.join(sparsity_by_exp, on="experiment")

# Sort columns: metadata → sparsity → metric@cutoff
cutoffs = sorted(long_df["cutoff"].unique())
metric_cols_ordered = [f"{m}@{c}" for c in cutoffs for m in MAIN_METRICS]
final_cols = (
    index_cols
    + sparsity_present
    + [c for c in metric_cols_ordered if c in wide_df.columns]
)
wide_df = wide_df[[c for c in final_cols if c in wide_df.columns]]

# Re-sort rows
wide_df["_strategy_order"] = wide_df["strategy"].map(_STRATEGY_ORDER).fillna(99)
wide_df = (
    wide_df
    .sort_values(["dataset", "_strategy_order", "keep_frac", "model"])
    .drop(columns=["_strategy_order"])
    .reset_index(drop=True)
)

print(f"wide_summary: {wide_df.shape}")
wide_df.head(4)

In [ ]:
# ── Sheet: main_metrics ───────────────────────────────────────────────────────

_meta_cols = [
    "experiment", "dataset", "strategy", "keep_frac", "k_user", "k_item",
    "n_interactions_x1000", "oss_pct", "uss", "iss",
    "user_gini", "item_gini",
    "coldstart_user_pct", "coldstart_item_pct",
    "cutoff", "model",
]

sheet_main = long_df[
    [c for c in _meta_cols if c in long_df.columns] + MAIN_METRICS
].copy()

print(f"main_metrics: {sheet_main.shape}")
sheet_main.head(3)

In [ ]:
# ── Sheet: u1_metrics ────────────────────────────────────────────────────────

_meta_cols_u1 = [
    "experiment", "dataset", "strategy", "keep_frac", "k_user", "k_item",
    "n_interactions_x1000", "oss_pct", "uss", "iss",
    "user_gini", "item_gini",
    "coldstart_user_pct", "coldstart_item_pct",
    "cutoff", "model",
]

sheet_u1 = long_df[
    [c for c in _meta_cols_u1 if c in long_df.columns] + U1_METRICS
].copy()

print(f"u1_metrics: {sheet_u1.shape}")
sheet_u1.head(3)

In [ ]:
# ── Write Excel ───────────────────────────────────────────────────────────────

output_path = Path(OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

    for df, sheet_name in [
        (sheet_long, "long_all_results"),
        (wide_df,    "wide_summary"),
        (sheet_main, "main_metrics"),
        (sheet_u1,   "u1_metrics"),
    ]:
        df.to_excel(writer, sheet_name=sheet_name, index=False)

        # Auto-width columns
        ws = writer.sheets[sheet_name]
        for col_cells in ws.columns:
            max_len = max(
                (len(str(cell.value)) if cell.value is not None else 0)
                for cell in col_cells
            )
            ws.column_dimensions[col_cells[0].column_letter].width = min(max_len + 2, 40)

        # Freeze header row
        ws.freeze_panes = "A2"

print(f"Written: {output_path}")
print(f"  long_all_results : {sheet_long.shape[0]} rows × {sheet_long.shape[1]} cols")
print(f"  wide_summary     : {wide_df.shape[0]} rows × {wide_df.shape[1]} cols")
print(f"  main_metrics     : {sheet_main.shape[0]} rows × {sheet_main.shape[1]} cols")
print(f"  u1_metrics       : {sheet_u1.shape[0]} rows × {sheet_u1.shape[1]} cols")

In [ ]:
# ── Preview: wide_summary ─────────────────────────────────────────────────────
# Xem nDCG@10 và Precision@10 cho tất cả experiments

preview_cols = ["dataset", "strategy", "keep_frac", "model"] + [
    c for c in wide_df.columns if "nDCG" in c or "Recall" in c
]
display_df = wide_df[[c for c in preview_cols if c in wide_df.columns]]

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)
display_df